# Clasificación de Deserción Estudiantil mediante SVM bajo Múltiples Métodos de Normalización

**Proyecto**: Machine Learning II - Universidad 2026-II  
**Autor**: Arnold Santiago Torres (`Arnold`)  
**Basado en**: *Support Vector Machine for Predicting Student Dropout Under Different Normalization Methods* (IEEE Xplore, 2024)  

---

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    MaxAbsScaler,
    QuantileTransformer,
    PowerTransformer,
    OrdinalEncoder,
    OneHotEncoder
)
from scipy.stats import ttest_ind

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='viridis')

## 1. Carga del Dataset y Preprocesamiento Base

In [ ]:
data_path = 'UCI_Dataset/data.csv'
df = pd.read_csv(data_path, sep=';')

# Crear etiqueta binaria: Dropout = 1, Graduate/Enrolled = 0
df['Target_Binary'] = df['Target'].apply(lambda x: 1 if x == 'Dropout' else 0)

X_raw = df.drop(columns=['Target', 'Target_Binary'])
y = df['Target_Binary'].values

print(f"Dimensiones: {df.shape}")
print("Distribución de Clases:")
print(pd.Series(y).value_counts(normalize=True))

## 2. Definición de Métodos de Normalización (Tabla I del Artículo)

In [ ]:
normalization_methods = {
    'Quantile Transformer': QuantileTransformer(n_quantiles=100, random_state=42, output_distribution='uniform'),
    'Power Transformer': PowerTransformer(method='yeo-johnson'),
    'One Hot Encoder': OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
    'Standard Scaler': StandardScaler(),
    'Min Max Scaler': MinMaxScaler(),
    'Max Absolute Scaler': MaxAbsScaler(),
    'Ordinal Encoder': OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),
    'No Normalization': None
}

## 3. Experimento de Evaluación de Desempeño SVM (Tabla II)

In [ ]:
N_TRIALS = 15
results = {method: [] for method in normalization_methods.keys()}

for trial in range(N_TRIALS):
    X_train, X_test, y_train, y_test = train_test_split(
        X_raw, y, test_size=0.2, random_state=42 + trial, stratify=y
    )
    for method_name, scaler in normalization_methods.items():
        try:
            if scaler is None:
                X_tr, X_te = X_train.values, X_test.values
            else:
                X_tr = scaler.fit_transform(X_train)
                X_te = scaler.transform(X_test)
            
            clf = SVC(kernel='rbf', C=1.0, cache_size=1000, random_state=42)
            clf.fit(X_tr, y_train)
            preds = clf.predict(X_te)
            results[method_name].append(f1_score(y_test, preds))
        except Exception as e:
            pass

## 4. Visualización de Resultados

In [ ]:
results_df = pd.DataFrame(results)
summary = pd.DataFrame({
    'Average F1 Score': results_df.mean(),
    'Std Dev': results_df.std()
}).sort_values('Average F1 Score', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=summary['Average F1 Score'], y=summary.index, palette='crest')
plt.title('Comparación de F1-Score Promedio según Método de Normalización (SVM)')
plt.xlabel('F1-Score Promedio')
plt.xlim(0, 1.0)
plt.show()
display(summary)